# 16 — EfficientNet-B0 | Transfer Learning
**Girdi:** RGB JPG görüntüler (HOG / Color özellikleri değil). Önceden eğitilmiş ImageNet ağırlıkları ilk çalıştırmada indirilebilir. Aynı image-level train/val/test ayrımı; test kullanılmaz. 2 aşama: önce sınıflandırıcı katmanı sonra seçili üst katmanlar fine-tuning. Modelin validation üzerinde seçimi sınırlı veriyle yapıldığından ihtiyatlı yorumlanmalı.

In [6]:
from pathlib import Path
import sys, json, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.config import BASE_DIR
from src.data_loader import load_dataset
FEATURE_DIR = BASE_DIR / "outputs" / "features"
RESULT_DIR = BASE_DIR / "outputs" / "models" / "deep_learning"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
df = load_dataset(require_masks=True).reset_index(drop=True)
y = df['label'].to_numpy(dtype=str)
ids = df['image'].to_numpy(dtype=str)
classes = np.unique(y)
idx = np.arange(len(y))
dev_idx, test_idx = train_test_split(idx, test_size=.15, random_state=42, stratify=y)
train_idx, val_idx = train_test_split(dev_idx, test_size=.15/.85, random_state=42, stratify=y[dev_idx])
assert len(set(train_idx)&set(val_idx)) == len(set(train_idx)&set(test_idx)) == len(set(val_idx)&set(test_idx)) == 0
print('Train:',len(train_idx),'Val:',len(val_idx),'Test (kullanılmayacak):',len(test_idx))
# ÖNEMLİ: Aynı lezyondan farklı görüntüler varsa lesion_id ile group split ayrıca uygulanmalı.


Train: 7009 Val: 1503 Test (kullanılmayacak): 1503


In [7]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
MODEL_NAME='EfficientNet_B0'
IMAGE_SIZE=225
BATCH_SIZE=16  # B5 çok daha fazla bellek ister; OOM olursa 2/1 yap
EPOCHS_HEAD=12
EPOCHS_FINE=12
UNFREEZE_LAST=25
ENCODER=LabelEncoder().fit(y)
yi=ENCODER.transform(y)
class_w=compute_class_weight('balanced',classes=np.arange(len(classes)),y=yi[train_idx])
class_weights={int(i):float(w) for i,w in enumerate(class_w)}
print('TensorFlow GPU:',tf.config.list_physical_devices('GPU'))


TensorFlow GPU: []


In [8]:
# tf.data: JPEG okuma ve float32 piksel [0,255]. EfficientNet dahili scaling içerir.
paths_all=df['image_path'].astype(str).to_numpy()
def make_dataset(indices,training):
    xx=paths_all[indices];yy=yi[indices]
    ds=tf.data.Dataset.from_tensor_slices((xx,yy))
    def decode(p,label):
        image=tf.io.decode_jpeg(tf.io.read_file(p),channels=3)
        #image=tf.image.resize(image,(224,224))
        image = tf.image.resize_with_pad(image, 225, 225)
        return image, label
    if training:ds=ds.shuffle(min(len(indices),2048),seed=42,reshuffle_each_iteration=True)
    ds=ds.map(decode,num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
train_ds=make_dataset(train_idx,True)
val_ds=make_dataset(val_idx,False)
# Tahminlerde sırayı korumak için ayrı shuffle=False train veri seti.
train_eval_ds=make_dataset(train_idx,False)


In [9]:
tf.keras.backend.clear_session();tf.keras.utils.set_random_seed(42)
backbone=tf.keras.applications.EfficientNetB0(include_top=False,weights='imagenet',input_shape=(IMAGE_SIZE,IMAGE_SIZE,3))
backbone.trainable=False
inp=layers.Input((IMAGE_SIZE,IMAGE_SIZE,3))
aug=tf.keras.Sequential([layers.RandomFlip('horizontal_and_vertical'),layers.RandomRotation(.08),layers.RandomZoom(.1)])
x=aug(inp)
x=backbone(x,training=False)
x=layers.GlobalAveragePooling2D()(x)
x=layers.Dropout(.4)(x)
out=layers.Dense(len(classes),activation='softmax')(x)
model=tf.keras.Model(inp,out)
prefix=RESULT_DIR/f'{MODEL_NAME}_original'
def compile_model(lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
def callbacks(suffix):
    return [EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True),
     ReduceLROnPlateau(monitor='val_loss',patience=2,factor=.5,min_lr=1e-7),
     ModelCheckpoint(str(prefix)+suffix+'.keras',monitor='val_loss',save_best_only=True)]
compile_model(1e-3)
start=time.time()
h1=model.fit(train_ds,validation_data=val_ds,epochs=EPOCHS_HEAD,class_weight=class_weights,callbacks=callbacks('_head'),verbose=1)
# Fine-tuning: BatchNormalization katmanlarını dondur, pretrained katmanları bozmamaya dikkat et.
backbone.trainable=True
for lyr in backbone.layers[:-UNFREEZE_LAST]:lyr.trainable=False
for lyr in backbone.layers:
    if isinstance(lyr,layers.BatchNormalization):lyr.trainable=False
compile_model(1e-5)
h2=model.fit(train_ds,validation_data=val_ds,epochs=EPOCHS_FINE,class_weight=class_weights,callbacks=callbacks('_fine'),verbose=1)
# İki aşama arasında en iyi doğrulama kaybını seç: final tek bir aşamaya bağımlı kalmasın.
a=min(h1.history['val_loss']);b=min(h2.history['val_loss'])
best_path=str(prefix)+('_head.keras' if a<=b else '_fine.keras')
model=tf.keras.models.load_model(best_path)
print('Seçilen model:',best_path, 'val_loss',min(a,b))


Epoch 1/12
439/439 ━━━━━━━━━━━━━━━━━━━━ 246s 538ms/step - accuracy: 0.3946 - loss: 1.7114 - val_accuracy: 0.4724 - val_loss: 1.4209 - learning_rate: 0.0010
Epoch 2/12
439/439 ━━━━━━━━━━━━━━━━━━━━ 233s 530ms/step - accuracy: 0.4908 - loss: 1.4073 - val_accuracy: 0.5848 - val_loss: 1.1661 - learning_rate: 0.0010
Epoch 3/12
439/439 ━━━━━━━━━━━━━━━━━━━━ 229s 522ms/step - accuracy: 0.5356 - loss: 1.3342 - val_accuracy: 0.6208 - val_loss: 1.0350 - learning_rate: 0.0010
Epoch 4/12
439/439 ━━━━━━━━━━━━━━━━━━━━ 226s 514ms/step - accuracy: 0.5390 - loss: 1.2562 - val_accuracy: 0.6667 - val_loss: 0.9251 - learning_rate: 0.0010
Epoch 5/12
439/439 ━━━━━━━━━━━━━━━━━━━━ 226s 514ms/step - accuracy: 0.5748 - loss: 1.2209 - val_accuracy: 0.6487 - val_loss: 0.9594 - learning_rate: 0.0010
Epoch 6/12
439/439 ━━━━━━━━━━━━━━━━━━━━ 232s 528ms/step - accuracy: 0.5707 - loss: 1.1866 - val_accuracy: 0.6434 - val_loss: 1.0159 - learning_rate: 0.0010
Epoch 7/12
439/439 ━━━━━━━━━━━━━━━━━━━━ 231s 525ms/step - accura

KeyboardInterrupt: 

In [ ]:
def store_result(model_name,variant,pred,train_pred,best_params,cv_score=None,seconds=None):
    prefix=RESULT_DIR/f'{model_name}_{variant}'
    r={'Model':model_name,'Variant':variant,'CV_Macro_F1':cv_score,
       'Train_Macro_F1':float(f1_score(y[train_idx],train_pred,average='macro',zero_division=0)),
       'Val_Accuracy':float(accuracy_score(y[val_idx],pred)),
       'Val_Macro_F1':float(f1_score(y[val_idx],pred,average='macro',zero_division=0)),
       'Val_Macro_Precision':float(precision_score(y[val_idx],pred,average='macro',zero_division=0)),
       'Val_Macro_Recall':float(recall_score(y[val_idx],pred,average='macro',zero_division=0)),
       'MEL_Recall':float(recall_score(y[val_idx],pred,labels=['MEL'],average='macro',zero_division=0)),
       'MEL_Precision':float(precision_score(y[val_idx],pred,labels=['MEL'],average='macro',zero_division=0)),
       'Predicted_Classes':int(len(np.unique(pred))),'Best_Params':json.dumps(best_params,ensure_ascii=False,default=str),
       'Search_Seconds':seconds,'Train_N':len(train_idx),'Val_N':len(val_idx),'Test_N':len(test_idx)}
    pd.DataFrame({'image_id':ids[val_idx], 'actual':y[val_idx], 'predicted':pred}).to_csv(str(prefix)+'_val_predictions1.csv',index=False)
    pd.DataFrame(confusion_matrix(y[val_idx],pred,labels=classes),index=classes,columns=classes).to_csv(str(prefix)+'_confusion_matrix1.csv')
    print(classification_report(y[val_idx],pred,labels=classes,zero_division=0))
    return r


In [ ]:
pred=ENCODER.inverse_transform(model.predict(val_ds,verbose=1).argmax(axis=1))
tr=ENCODER.inverse_transform(model.predict(train_eval_ds,verbose=1).argmax(axis=1))
import joblib
joblib.dump(ENCODER,str(prefix)+'_label_encoder1.joblib')
pd.DataFrame(h1.history).to_csv(str(prefix)+'_head_history1.csv',index=False)
pd.DataFrame(h2.history).to_csv(str(prefix)+'_fine_history1.csv',index=False)
result=store_result(MODEL_NAME,'original',pred,tr,{'architecture':'B0_1','batch_size':BATCH_SIZE,'head_epochs':len(h1.history['loss']),'fine_epochs':len(h2.history['loss'])},None,round(time.time()-start,1))
pd.DataFrame([result]).to_csv(RESULT_DIR/f'{MODEL_NAME}_summary1.csv',index=False)
display(pd.DataFrame([result]))
print('Test seti kullanılmadı. Image-level split veri sızıntısı riski taşıyabilir.')


In [ ]:
import winsound

winsound.Beep(300, 3000)
print("Ses testi tamamlandı.")